Approach 4: retrieval and label conditioned generation
Owner: Yousef

In [ ]:
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Google Drive mount skipped (not running in Colab).")

# The checked-out working folder is primary; GitHub is only a fresh-Colab fallback.
CANDIDATES = [Path.cwd(), Path("/content/cxr_second_reader"), Path("/content/drive/MyDrive/cxr_second_reader")]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "config.py").exists()), None)
if REPO_ROOT is None:
    if not Path("/content").exists():
        raise FileNotFoundError("Open this notebook from the cxr_second_reader project folder.")
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/eyadstein/cxr_second_reader.git", "/content/cxr_second_reader"], check=True)
    REPO_ROOT = Path("/content/cxr_second_reader")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Project folder:", REPO_ROOT)


In [ ]:
from src.data import load_table, CXRDataset
train_df = load_table("train")
val_df = load_table("val")
test_df = load_table("test")
print(f"Studies/rows — train: {len(train_df)}, validation: {len(val_df)}, test: {len(test_df)}")
print("Columns:", train_df.columns.tolist())
print("Target example:", train_df.iloc[0]["target_text"][:300])


In [ ]:
# Text-only retrieval control: fit exclusively on training reports; do not treat
# this as image retrieval or a clinically validated report generator.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from src.evaluate import evaluate, log_result

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=50000)
train_text = train_df["target_text"].fillna("").astype(str).tolist()
test_text = test_df["target_text"].fillna("").astype(str).tolist()
train_matrix = vectorizer.fit_transform(train_text)
test_matrix = vectorizer.transform(test_text)
similarity = cosine_similarity(test_matrix, train_matrix)
retrieved = [train_text[i] for i in similarity.argmax(axis=1)]
print("Test rows with a retrieved training report:", len(retrieved))
print("Mean best-match cosine similarity:", round(float(similarity.max(axis=1).mean()), 4))


In [ ]:
metrics = evaluate(retrieved, test_text)
log_result("05_text_tfidf_retrieval_control", metrics)
print(metrics)
